# OOO30 vs CT30 Month-End Backtest

This notebook runs the simple month-end cash-bond strategy described here, not the exact auction-cycle trade in Krishnamurthy (2002). The backtest keeps bond PnL on `NPV`, applies a reusable FRB financing layer with `GC + specialness`, and plots gross bond/coupon PnL separately from financing PnL.


In [ ]:
%load_ext autoreload
%autoreload 2

import datetime
import sys

import matplotlib.pyplot as plt
import matplotlib.pylab as pylab
import pandas as pd
import QuantLib as ql

plt.style.use("ggplot")
pylab.rcParams.update(
    {
        "legend.fontsize": "x-large",
        "figure.figsize": (14, 10),
        "axes.labelsize": "x-large",
        "axes.titlesize": "x-large",
        "xtick.labelsize": "large",
        "ytick.labelsize": "large",
    }
)

sys.path.append("../../")

from BT.data_handler import TimeGrid
from BT.misc import (
    _last_business_day_of_month,
    _month_iter,
    _n_business_days_before,
    _nth_business_day_of_month,
    ql_cal_date_range,
)
from BT.query_actions import AddQueryAction, UnwindPositionsAction
from BT.query_engine import QueryDrivenBacktest
from BT.query_strategy import QueryStrategy
from BT.triggers import DateTrigger, DateTriggerRequirements
from MDP.FixedRateBonds.FixedRateBondsMDP import FixedRateBondsMDP
from Query.Base.query_resolution import resolve_query
from Query.FixedRateBonds.carry_roll import load_us_treasury_gc_fixing_pct
from Query.FixedRateBonds.FixedRateBondQuery import FixedRateBondQuery
from Query.FixedRateBonds.FixedRateBondValue import FixedRateBondValue


In [ ]:
CAL = ql.UnitedStates(ql.UnitedStates.GovernmentBond)
risk_bpv = -100_000

start = datetime.date(2023, 1, 1)
end = datetime.date(2026, 3, 20)

days_pre_month_end = 4
days_after_month_end = 1

cycles = []
for y, m in _month_iter(start, end):
    eom_bd = _last_business_day_of_month(CAL, y, m)
    entry = _n_business_days_before(CAL, eom_bd, days_pre_month_end)
    ny, nm = (y + 1, 1) if m == 12 else (y, m + 1)
    exit_ = _nth_business_day_of_month(CAL, ny, nm, days_after_month_end)

    if entry < start or exit_ > end:
        continue

    cycles.append((entry, exit_, f"me-trade-{y:04d}{m:02d}"))

tg = TimeGrid(
    ql_cal_date_range(
        ql_cal=CAL,
        start=min(entry for entry, _, _ in cycles),
        end=max(exit_ for _, exit_, _ in cycles),
    )
)

gc_fixing_pct = pd.Series(
    {
        d: load_us_treasury_gc_fixing_pct(d)
        for d in sorted({ts.date() for ts in tg})
    },
    dtype=float,
).sort_index()

if gc_fixing_pct.isna().any():
    missing = gc_fixing_pct[gc_fixing_pct.isna()].index.tolist()
    raise ValueError(f"Missing GC fixing data for backtest dates: {missing[:10]}")

financing_config = {
    "mode": "gc_plus_specialness",
    "gc_rate": gc_fixing_pct / 100.0,
    "leg_specialness_bps": {"front": 0.0, "back": 100.0},
    "day_count": "ACT/360",
    "haircut": 0.0,
}

frb_mdp = FixedRateBondsMDP(source="USTS_FEDINVEST_WSJ_LIVE-QL")
trade_query = FixedRateBondQuery(
    cusip="ooo30/CT30",
    value=FixedRateBondValue.NPV,
    structure_kwargs={"bpv": risk_bpv},
    meta={"financing": financing_config},
)

len(cycles), cycles[:3]


In [ ]:
triggers = []
for entry_date, exit_date, tag in cycles:
    tagged_query = FixedRateBondQuery(
        cusip=trade_query.cusip,
        value=trade_query.value,
        structure_kwargs=dict(trade_query.structure_kwargs),
        meta=dict(trade_query.meta),
        tags=(tag,),
    )
    triggers.extend(
        [
            DateTrigger(DateTriggerRequirements(dates=[entry_date]), actions=[AddQueryAction(query=tagged_query)]),
            DateTrigger(
                DateTriggerRequirements(dates=[exit_date]),
                actions=[UnwindPositionsAction(match_tag=tag, fee=0.0)],
            ),
        ]
    )

strategy = QueryStrategy(name="Month-End OOO30 vs CT30", triggers=triggers)
bt = QueryDrivenBacktest(time_grid=tg, mdp=frb_mdp, strategy=strategy)
bt.run()


In [ ]:
time_index = pd.Index(list(tg), name="date")
component_histories = getattr(bt, "frb_component_histories", {})

gross_bond = pd.Series(component_histories.get("bond_total", {}), dtype=float).sort_index()
financing = pd.Series(component_histories.get("financing_total", {}), dtype=float).sort_index()
net = pd.Series(component_histories.get("net_total", {}), dtype=float).sort_index()

gross_bond = gross_bond.reindex(time_index).ffill().fillna(0.0)
financing = financing.reindex(time_index).ffill().fillna(0.0)
net = net.reindex(time_index).ffill().fillna(0.0)

resolved_rows = []
for entry_date, exit_date, tag in cycles:
    cycle_query = FixedRateBondQuery(
        cusip=trade_query.cusip,
        value=trade_query.value,
        structure_kwargs=dict(trade_query.structure_kwargs),
        meta=dict(trade_query.meta),
        tags=(tag,),
    )
    cycle_ts = datetime.datetime.combine(entry_date, datetime.time())
    cycle_pricers = frb_mdp.get_pricer(cycle_query.build_mdp_request(cycle_ts))
    cycle_resolved = resolve_query(cycle_query, timestamp=cycle_ts, pricer_or_curve=cycle_pricers)
    cycle_package, cycle_weights = cycle_resolved.resolve_package(pricer_or_curve=cycle_pricers)
    resolved_rows.append(
        {
            "tag": tag,
            "entry_date": entry_date,
            "exit_date": exit_date,
            "front_cusip": cycle_package[0].cusip,
            "back_cusip": cycle_package[1].cusip,
            "front_notional": cycle_package[0].notional,
            "back_notional": cycle_package[1].notional,
            "front_weight": cycle_weights[0],
            "back_weight": cycle_weights[1],
        }
    )

resolved_legs = pd.DataFrame(resolved_rows)

dirty_rows = []
for ts in time_index:
    dirty_query = FixedRateBondQuery(
        cusip=trade_query.cusip,
        value=FixedRateBondValue.DIRTY_PRICE,
        structure_kwargs=dict(trade_query.structure_kwargs),
    )
    pricers = frb_mdp.get_pricer(dirty_query.build_mdp_request(ts))
    resolved = resolve_query(dirty_query, timestamp=ts, pricer_or_curve=pricers)
    package, _ = resolved.resolve_package(pricer_or_curve=pricers)
    front_pricer = pricers[package[0].cusip]
    back_pricer = pricers[package[1].cusip]
    front_dirty = float(front_pricer.dirty_price())
    back_dirty = float(back_pricer.dirty_price())
    dirty_rows.append(
        {
            "date": ts,
            "front_cusip": package[0].cusip,
            "back_cusip": package[1].cusip,
            "ooo30_dirty_per_100": front_dirty,
            "ct30_dirty_per_100": back_dirty,
            "dirty_spread_per_100": front_dirty - back_dirty,
        }
    )

dirty_spread = pd.DataFrame(dirty_rows).set_index("date").sort_index()

summary = pd.DataFrame(
    [
        {
            "gross_bond_coupon_pnl": float(gross_bond.iloc[-1]),
            "financing_pnl": float(financing.iloc[-1]),
            "net_pnl": float(net.iloc[-1]),
        }
    ]
)

summary, resolved_legs.head(10)


In [ ]:
fig, axes = plt.subplots(2, 1, sharex=True, figsize=(14, 10))

axes[0].plot(gross_bond.index, gross_bond.values, label="Gross bond/coupon PnL")
axes[0].plot(financing.index, financing.values, label="Financing PnL")
axes[0].plot(net.index, net.values, label="Net PnL", linewidth=2)
axes[0].set_title("OOO30/CT30 Month-End Backtest: Gross, Financing, Net")
axes[0].set_ylabel("PnL")
axes[0].legend(loc="best")

axes[1].plot(dirty_spread.index, dirty_spread["dirty_spread_per_100"].values, color="black")
axes[1].set_title("OOO30 - CT30 Dirty Price Spread (descriptive only)")
axes[1].set_ylabel("Dirty price spread per 100")
axes[1].set_xlabel("Date")

plt.tight_layout()
plt.show()
